# Healthy Cohort Extraction

This notebook extracts healthy control visits from the full DELCODE dataset.
Healthy rows are highlighted in **green** in the output Excel file.

## Exclusion Criteria

| Step | Rule | Source |
|---|---|---|
| 1. Exclude SCD | Remove rows where `prmdiag == 1` | Data Dictionary: `prmdiag` 1 = SCD (Subjektive kognitive Verschlechterung) |
| 2. Exclude MCI | Remove rows whose `Repseudonym` appears in `MCI_all_visits.csv` | MCI cohort (490 patients) |
| 3. Exclude AD | Remove rows whose `Repseudonym` appears in `AD_use_only_cognitive_scores_all_visits.csv` | AD cohort (128 patients) |

## Healthy Inclusion Criteria

| Criterion | Rule | Note |
|---|---|---|
| Cognitive (required) | `mmstot > 27` AND `cdrglobal == 0` | Always required |
| Biomarker (if available) | `ratio_Abeta42_40 >= 0.0806` AND `ratio_Abeta42_phosphotau181 >= 9.6825` | Rows with missing biomarkers are included |

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

# ── Centralized paths ──────────────────────────────────────────────────
BASE = Path('/dss/dsshome1/0A/di54lup/DELCODE/STRATIFICATION')
INTERMEDIATE = BASE / 'data' / 'intermediate'

full_input_csv = INTERMEDIATE / 'combined' / 'all_visits.csv'
mci_csv        = INTERMEDIATE / 'mci' / 'all_visits.csv'
ad_csv         = INTERMEDIATE / 'ad'  / 'cognitive_all_visits.csv'

output_dir  = INTERMEDIATE / 'healthy'
output_csv  = output_dir / 'all_visits.csv'
output_xlsx = output_dir / 'highlighted.xlsx'
output_dir.mkdir(parents=True, exist_ok=True)

# Load data
full = pd.read_csv(full_input_csv)
mci  = pd.read_csv(mci_csv)
ad   = pd.read_csv(ad_csv)

# Convert numeric columns
for col in ['mmstot', 'cdrglobal', 'ratio_Abeta42_40', 'ratio_Abeta42_phosphotau181']:
    full[col] = pd.to_numeric(full[col], errors='coerce')

print(f'Full dataset:   {len(full):,} rows, {full["Repseudonym"].nunique()} patients')
print(f'MCI exclusion:  {mci["Repseudonym"].nunique()} patients (from {mci_csv.name})')
print(f'AD exclusion:   {ad["Repseudonym"].nunique()} patients (from {ad_csv.name})')
print(f'\nprmdiag distribution in full dataset:')
print(full['prmdiag'].value_counts().sort_index().to_string())


Full dataset:   4,605 rows, 866 patients
MCI exclusion:  490 patients (from all_visits.csv)
AD exclusion:   108 patients (from cognitive_all_visits.csv)

prmdiag distribution in full dataset:
prmdiag
0    1396
1    2166
2     714
5     329


In [4]:
# ── Step 1: Exclude SCD (prmdiag == 1) ───────────────────────────────
n_before = len(full)
scd_patients = full[full['prmdiag'] == 1]['Repseudonym'].nunique()
scd_rows = (full['prmdiag'] == 1).sum()

df = full[full['prmdiag'] != 1].copy()

print(f'SCD rows removed:     {scd_rows:,} ({scd_patients} patients)')
print(f'Remaining:            {len(df):,} rows, {df["Repseudonym"].nunique()} patients')
print(f'Remaining prmdiag:    {df["prmdiag"].value_counts().sort_index().to_dict()}')

SCD rows removed:     2,166 (381 patients)
Remaining:            2,439 rows, 485 patients
Remaining prmdiag:    {0: 1396, 2: 714, 5: 329}


In [5]:
# ── Step 2 & 3: Exclude MCI and AD pseudonyms ────────────────────────
mci_pseudonyms = set(mci['Repseudonym'].unique())
ad_pseudonyms  = set(ad['Repseudonym'].unique())
exclude_pseudonyms = mci_pseudonyms | ad_pseudonyms

n_before_excl = len(df)
patients_before_excl = df['Repseudonym'].nunique()

df = df[~df['Repseudonym'].isin(exclude_pseudonyms)].copy()

print(f'MCI pseudonyms to exclude:  {len(mci_pseudonyms)}')
print(f'AD pseudonyms to exclude:   {len(ad_pseudonyms)}')
print(f'Combined (union):           {len(exclude_pseudonyms)}')
print(f'\nRows before exclusion:      {n_before_excl:,} ({patients_before_excl} patients)')
print(f'Rows after exclusion:       {len(df):,} ({df["Repseudonym"].nunique()} patients)')
print(f'Rows removed:               {n_before_excl - len(df):,}')

# Verify zero overlap
overlap = set(df['Repseudonym']) & exclude_pseudonyms
assert len(overlap) == 0, f'Overlap found: {overlap}'
print(f'\n✓ Zero overlap with MCI/AD patients confirmed')

MCI pseudonyms to exclude:  490
AD pseudonyms to exclude:   108
Combined (union):           559

Rows before exclusion:      2,439 (485 patients)
Rows after exclusion:       1,098 (196 patients)
Rows removed:               1,341

✓ Zero overlap with MCI/AD patients confirmed


In [6]:
# ── Step 4: Apply healthy criteria ───────────────────────────────────

# Cognitive criteria (always required)
cog_ok = (df['mmstot'] > 27) & (df['cdrglobal'] == 0)

# Biomarker criteria: only fail if biomarkers are PRESENT and ABNORMAL
bio_ab_available  = df['ratio_Abeta42_40'].notna()
bio_ptau_available = df['ratio_Abeta42_phosphotau181'].notna()

bio_ab_ok   = ~bio_ab_available  | (df['ratio_Abeta42_40'] >= 0.0806)
bio_ptau_ok = ~bio_ptau_available | (df['ratio_Abeta42_phosphotau181'] >= 9.6825)

healthy = cog_ok & bio_ab_ok & bio_ptau_ok

print(f'After SCD/MCI/AD exclusion: {len(df):,} rows')
print(f'\nCognitive criteria met (MMSE>27, CDR=0):  {cog_ok.sum()}')
print(f'  Of those, biomarkers available:          {(cog_ok & bio_ab_available).sum()}')
print(f'  Of those, biomarkers normal:             {(cog_ok & bio_ab_available & bio_ab_ok & bio_ptau_ok).sum()}')
print(f'  Of those, biomarkers abnormal (excl.):   {(cog_ok & bio_ab_available & ~(bio_ab_ok & bio_ptau_ok)).sum()}')
print(f'\nTotal healthy rows:    {healthy.sum()}')
print(f'Healthy patients:      {df.loc[healthy, "Repseudonym"].nunique()}')

df_healthy = df[healthy].copy()

# Final assertions
assert (df_healthy['prmdiag'] == 1).sum() == 0, 'SCD rows found in output!'
assert len(set(df_healthy['Repseudonym']) & mci_pseudonyms) == 0, 'MCI overlap!'
assert len(set(df_healthy['Repseudonym']) & ad_pseudonyms) == 0, 'AD overlap!'
print('\n✓ All assertions passed (no SCD, no MCI, no AD)')

After SCD/MCI/AD exclusion: 1,098 rows

Cognitive criteria met (MMSE>27, CDR=0):  987
  Of those, biomarkers available:          138
  Of those, biomarkers normal:             115
  Of those, biomarkers abnormal (excl.):   23

Total healthy rows:    964
Healthy patients:      176

✓ All assertions passed (no SCD, no MCI, no AD)


In [7]:
# ── Step 5: Save CSV + highlighted Excel ─────────────────────────────
from openpyxl.styles import PatternFill

GREEN_FILL = PatternFill(start_color='92D050', end_color='92D050', fill_type='solid')

# Save CSV
df_healthy.to_csv(output_csv, index=False)
print(f'✓ CSV saved: {output_csv}')
print(f'  Rows: {len(df_healthy):,}  |  Patients: {df_healthy["Repseudonym"].nunique()}')

# Save highlighted Excel — write the FULL dataset, highlight healthy rows green
display_cols = ['file', 'Repseudonym', 'visdate', 'scan_date', 'scan_year', 'prmdiag',
                'mmstot', 'cdrglobal', 'ratio_Abeta42_40',
                'ratio_Abeta42_phosphotau181', 'visnam']
display_cols = [c for c in display_cols if c in full.columns]

# Build display DataFrame from full dataset (after SCD/MCI/AD exclusion only)
# so we show all remaining rows but highlight the ones meeting healthy criteria
df_display = df[display_cols].copy()

# Track which rows in df are healthy
healthy_indices = set(df_healthy.index)

with pd.ExcelWriter(output_xlsx, engine='openpyxl') as writer:
    df_display.to_excel(writer, index=False, sheet_name='Healthy')
    ws = writer.sheets['Healthy']
    
    # Highlight healthy rows in green (row 1 = header, data starts at row 2)
    n_highlighted = 0
    for excel_row_idx, df_idx in enumerate(df_display.index, start=2):
        if df_idx in healthy_indices:
            for col_idx in range(1, len(display_cols) + 1):
                ws.cell(row=excel_row_idx, column=col_idx).fill = GREEN_FILL
            n_highlighted += 1

print(f'\n✓ Excel saved: {output_xlsx}')
print(f'  Total rows in sheet: {len(df_display):,}')
print(f'  Rows highlighted green: {n_highlighted:,}')
print(f'  File size: {output_xlsx.stat().st_size:,} bytes')

✓ CSV saved: /dss/dsshome1/0A/di54lup/DELCODE/STRATIFICATION/data/intermediate/healthy/all_visits.csv
  Rows: 964  |  Patients: 176

✓ Excel saved: /dss/dsshome1/0A/di54lup/DELCODE/STRATIFICATION/data/intermediate/healthy/highlighted.xlsx
  Total rows in sheet: 1,098
  Rows highlighted green: 964
  File size: 60,714 bytes


In [8]:
# ── Summary ──────────────────────────────────────────────────────────
print('=' * 70)
print('HEALTHY COHORT EXTRACTION SUMMARY')
print('=' * 70)
print(f'\nInput:  {full_input_csv.name} ({len(full):,} rows, {full["Repseudonym"].nunique()} patients)')
print(f'\nExclusions:')
print(f'  SCD (prmdiag == 1):  {scd_rows:,} rows, {scd_patients} patients')
print(f'  MCI pseudonyms:      {len(mci_pseudonyms)} patients (from {mci_csv.name})')
print(f'  AD pseudonyms:       {len(ad_pseudonyms)} patients (from {ad_csv.name})')
print(f'\nHealthy criteria:')
print(f'  Cognitive:  MMSE > 27 AND CDR = 0')
print(f'  Biomarker:  Aβ42/Aβ40 ≥ 0.0806 AND Aβ42/p-tau ≥ 9.6825 (if available)')
print(f'\nResults:')
print(f'  Healthy visits:   {len(df_healthy):,}')
print(f'  Healthy patients: {df_healthy["Repseudonym"].nunique()}')
print(f'  Avg visits/patient: {len(df_healthy) / max(df_healthy["Repseudonym"].nunique(), 1):.1f}')

has_bio = df_healthy['ratio_Abeta42_40'].notna().sum()
no_bio  = df_healthy['ratio_Abeta42_40'].isna().sum()
print(f'\n  With biomarkers (all normal):    {has_bio}')
print(f'  Without biomarkers (cog. only):  {no_bio}')
print(f'\nOutputs:')
print(f'  {output_csv}')
print(f'  {output_xlsx}')

HEALTHY COHORT EXTRACTION SUMMARY

Input:  all_visits.csv (4,605 rows, 866 patients)

Exclusions:
  SCD (prmdiag == 1):  2,166 rows, 381 patients
  MCI pseudonyms:      490 patients (from all_visits.csv)
  AD pseudonyms:       108 patients (from cognitive_all_visits.csv)

Healthy criteria:
  Cognitive:  MMSE > 27 AND CDR = 0
  Biomarker:  Aβ42/Aβ40 ≥ 0.0806 AND Aβ42/p-tau ≥ 9.6825 (if available)

Results:
  Healthy visits:   964
  Healthy patients: 176
  Avg visits/patient: 5.5

  With biomarkers (all normal):    115
  Without biomarkers (cog. only):  849

Outputs:
  /dss/dsshome1/0A/di54lup/DELCODE/STRATIFICATION/data/intermediate/healthy/all_visits.csv
  /dss/dsshome1/0A/di54lup/DELCODE/STRATIFICATION/data/intermediate/healthy/highlighted.xlsx
